# Earliest-Deadline-First Baseline — With-Reliability Group

Scheduling and discrete-time simulation definitions for the corresponding experimental comparison group. Generated outputs are written outside the notebook.

In [ ]:
# Imports
import gym
from gym import spaces
import pandas as pd
import numpy as np
import random
import math
from collections import deque
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d 
import os
from itertools import zip_longest
from scipy.stats import truncnorm
import csv
from functools import reduce
import torch
import torch.nn as nn
import copy

# Least common multiple
def lcm(a, b):
    return abs(a * b) // math.gcd(a, b)

def lcm_of_list(numbers):
    return reduce(lcm, numbers)


In [ ]:
# Shared scheduling state
class Shared:
    def __init__(self):
        self.slist = []
        self.is_primary_set = False


In [ ]:
class Task:
    def __init__(self, name, vf, initial_exec_task_time, deadline, alpha_c, I0_exp_term, peak_aC, min_aC, shared, WCET, worst_vf, exec_set, vf_levels, worst_lvl):
        self.name = name
        self.init_v, self.init_f = vf
        self.initial_exec_task_time = initial_exec_task_time
        self.current_exec_time = initial_exec_task_time
        self.alpha_C = alpha_c
        self.I0_exp_term = I0_exp_term
        self.peak_aC = peak_aC 
        self.min_aC = min_aC
        self.period = deadline
        self.deadline = deadline
        self.shared_tasks = shared
        self.arrival = 0
        self.fault = False
        self.WCET = WCET
        self.worst_v, self.worst_f = worst_vf
        self.exec_set = exec_set
        self.vf_levels = vf_levels
        self.v, self.f = vf
        self.core = None
        self.is_primary = False
        self.job_id = 0 
        self.slack = 0
        self.current_job_id = -1

        self.worst_lvl = worst_lvl

    def get_util(self):
        return (self.current_exec_time/self.period)
    def update(self, v, f, core):
        self.f = f
        self.v = v
        self.core = core
        self.current_exec_time = math.ceil((self.initial_exec_task_time / (f/self.init_f)))
    
    def get_peak_p(self):
        return (self.peak_aC*(self.v**2)*self.f + self.I0_exp_term * self.v)
    def get_min_p(self):
        return (self.min_aC*(self.v**2)*self.f + self.I0_exp_term * self.v)
    def get_avg_p(self):
        return (self.alpha_C*(self.v**2)*self.f + self.I0_exp_term * self.v)
    
    def get_vf_lvl_peak(self, i):
        return self.peak_aC*(self.vf_levels[i]['voltage']**2)*self.vf_levels[i]['frequency'] + self.I0_exp_term * self.vf_levels[i]['voltage']


In [ ]:
class Job(Task):
    def __init__(self, task: Task, shared):
        super().__init__(
            name=task.name,
            vf=(task.init_v, task.init_f),
            initial_exec_task_time=task.initial_exec_task_time,
            deadline=task.deadline,
            alpha_c=task.alpha_C,
            I0_exp_term=task.I0_exp_term,
            peak_aC=task.peak_aC,
            min_aC=task.min_aC,
            shared=shared,
            WCET=task.WCET,
            worst_vf=(task.worst_v, task.worst_f),
            exec_set=task.exec_set,
            vf_levels=task.vf_levels,
            worst_lvl=task.worst_lvl
        )
        self.min_start_time = -1
        self.vf_is_set = False
        self.real_exec = 0
        self.is_real_exec_set = False
        self.start_time = -1
        self.finish_time = -1
        self.current_peak = -1
        self.reliability = -1
        self.real_reliability = -1

    def job_update(self, task: Task):
        self.task = task
        self.arrival = task.arrival
        self.fault = task.fault
        self.v, self.f = task.v, task.f
        self.v_policy, self.f_policy = task.v, task.f
        self.core = task.core
        self.is_primary = task.is_primary
        self.current_exec_time = task.current_exec_time
        self.job_id = task.job_id

    def apply_DVFS(self, v, f):
        self.task.f = f
        self.task.v = v
        self.f = f
        self.v = v
        self.current_exec_time = math.ceil((self.task.initial_exec_task_time / (f/self.task.init_f)))
        self.task.current_exec_time = self.current_exec_time
    
    def get_current_util(self):
        return self.task.current_exec_time/self.task.period

    def get_current_peak(self):
        return (self.task.peak_aC*(self.task.v**2)*self.task.f + self.task.I0_exp_term * self.task.v)

    def get_rep_num(self):
        return (len(self.shared_tasks.slist) - 1)


In [ ]:
class Core:
    def __init__(self, id): 
        self.id = id
        self.ready = []
        self.running_replica = []
        self.mapped_for_slack = []
        self.primary = []
        self.replica_st = []
        self.replica_dl = []
        self.current_utilization = 0
        self.utilization = 0
        self.slack_time = 0
        self.running = None
    
    def EDF(self):
        self.primary.sort(key=lambda t:t.deadline)
        self.replica_st.sort(key=lambda t:t.min_start_time)
        self.replica_dl.sort(key=lambda t:t.deadline)
    
    def remove_replica(self, j):
        exist = False 
        self.replica_dl.remove(j)
        self.replica_st.remove(j)
        self.ready.remove(j)
        if self.running == j:
            self.running=None
            exist = True
            # current_TDP[0] -= j.get_peak_p()
        return exist
    
    def remove_primary(self, j):
        self.primary.remove(j)
        self.ready.remove(j)
        if self.running == j:
            self.running=None
            # current_TDP[0] -= j.get_peak_p()


In [ ]:
# Quality-of-service metric

def utility(fi, Di, x=2):
    fi = np.asarray(fi, dtype=float)
    Di = np.asarray(Di, dtype=float)
    if x <= 1:
        raise ValueError("x must be > 1")
    UF = np.zeros_like(fi, dtype=float)
    on_time = fi <= Di
    soft = (fi > Di) & (fi <= x * Di)
    UF[on_time] = 1.0
    UF[soft] = (Di[soft] - fi[soft]) / (Di[soft] * (x - 1)) + 1.0
    return UF

def qos_system_no_backup(P, f_main, D, x=2):
    P = np.asarray(P, dtype=float)
    f_main = np.asarray(f_main, dtype=float)
    D = np.asarray(D, dtype=float)
    if not (len(P) == len(f_main) == len(D)):
        raise ValueError("P, f_main, and D must have the same length")
    if np.any((P < 0) | (P > 1)):
        raise ValueError("Probabilities must be in [0,1]")
    UF = utility(f_main, D, x)
    qos = np.sum(P * UF) / len(P)
    return float(qos), UF


In [ ]:
# Scheduling environment
class Envir(gym.Env):
    def __init__(self, initial_voltage, initial_frequency, cores, main_tasks, alpha):
        super().__init__()
        self.cores = cores
        self.main_tasks = main_tasks
        self.current_task_count = 0
        self.static = 0
        self.alpha = alpha
        self.data = []
        self.train_data = []
        # VF levels: frequency in Hz, voltage in V
        self.vf_levels = {
            0: {"frequency": 800e6, "voltage": 0.9},
            1: {"frequency": 1.1e9, "voltage": 0.95},
            2: {"frequency": 1.4e9, "voltage": 1.0},
            3: {"frequency": 1.7e9, "voltage": 1.05},
            4: {"frequency": 2.0e9, "voltage": 1.1},
        }
        self.initial_voltage = self.vf_levels[len(self.vf_levels)-1]['voltage']
        self.initial_frequency = self.vf_levels[len(self.vf_levels)-1]['frequency']
        self.lambda_0 = 1e-7  # faults/usec
        self.Delta = 1.0      # volt
        self.V_max = self.vf_levels[4]['voltage']  # max voltage

        self.system_tasks = []
        self.state = None
        self.action = None
        self.total_util = 0

        self.reset()

    def reset(self):
        self.state = {core: 0.0 for core in range(self.cores)}
        self.action = None
        self.total_util = 0
        self.train_data = self.data.copy()
        self.current_task_count = 0
        self.system_tasks.clear()
        return self.state

    def read_data(self, input_files):
        ds = []
        try:
            # def __init__(self, name, vf, initial_exec_task_time, deadline, alpha_c, I0_exp_term, peak_aC, static, min_aC):
            for file in input_files:
                
                task_name = os.path.basename(file).replace('.csv', '')
                task_data = pd.read_csv(file)
                
                initial_exec_task_time = task_data["#"].iloc[0]
                self.static = task_data["static"].iloc[0]
                total_power = task_data["avg_power"].iloc[0]
                alpha_C = task_data["alpha_C"].iloc[0]
                I0_exp_term = task_data["I0_exp_term"].iloc[0]
                util = task_data["util"].iloc[0]
                peak = task_data["peak"].iloc[0]
                deadline = task_data["deadline"].iloc[0]
                exec_set = {i:0 for i in range(len(self.vf_levels))}
                peak_list = []
                worst_lvl = -1
                for i in range(len(self.vf_levels)):
                    exec_set[i] = math.ceil(initial_exec_task_time / (self.vf_levels[i]['frequency']/self.initial_frequency))
                    peak_list.append((peak*(self.vf_levels[i]['voltage']**2)*self.vf_levels[i]['frequency'] + I0_exp_term*self.vf_levels[i]['voltage'], i))
                peak_list.sort(key=lambda t:t[0], reverse=True)
                for tup in peak_list:
                    if tup[0] <= TSP:
                        worst_lvl = tup[1]
                        break
                WCET = math.ceil(initial_exec_task_time / (self.vf_levels[worst_lvl]['frequency']/self.initial_frequency))
                
                #     exec_set[i] = math.ceil(initial_exec_task_time / (self.vf_levels[i]['frequency']/self.initial_frequency))
                self.data.append(Task(task_name, (self.initial_voltage, self.initial_frequency), 
                            initial_exec_task_time, deadline, alpha_C, I0_exp_term, peak, 
                            task_data["min"].head(1).mean(), Shared(), WCET, (self.vf_levels[worst_lvl]['voltage'], self.vf_levels[worst_lvl]['frequency']),
                            exec_set, self.vf_levels, worst_lvl
                            ))

            #     ds.append(z.deadline)
        except Exception as e:
            print(f"An error occurred: {e}")


In [ ]:
# Slack calculation
def calculate_slack(cores, hyper_period):
    for c in cores:
        total_WCET = 0
        for job in c.mapped_for_slack:
            total_WCET += job.WCET
        c.slack_time = hyper_period - total_WCET

def define_slack(cores, data, hyper_period):
    for c in cores:
        w_t = []
        for d in data:
            if c != d.core:
                continue
            w_t.append((hyper_period/d.period)*d.WCET)
        w_t = sum(w_t)
        for d in data:
            if c != d.core:
                continue
            d.slack = int((((hyper_period/d.period)*d.WCET)/w_t)*c.slack_time)


In [ ]:
# Mapping and job creation
def map(data, ustate, env, cores):
        # primary
    for task in data:
        min_index = min(ustate, key=ustate.get)
        core = cores[min_index]
        task.update(env.vf_levels[len(env.vf_levels)-1]["voltage"], env.vf_levels[len(env.vf_levels)-1]["frequency"], core)
        ustate[min_index] += task.get_util()
        task.shared_tasks.slist.append(task)
        task.core.current_utilization += task.get_util()

    #     s[d.core.id] = 1e4
        
    #     ustate[min_index] += backup_d.get_util()
    #     backup_d.shared_tasks.slist.append(backup_d)

# Job creation
def create_jobs(data, hyper_period, cores, env):
    jobs = []
    arrivals = []
    for t in range(hyper_period):
        for task in data:
            if task.arrival == t:
                p_job = Job(task, Shared())
                p_job.job_update(task)
                p_job.shared_tasks.slist.append(p_job)
                p_job.job_id = task.job_id
                jobs.append(p_job)
                arrivals.append(task.arrival)
                p_job.core.mapped_for_slack.append(p_job)
                p_job.is_primary = True
                for rep in task.shared_tasks.slist:
                    if rep == task: continue
                    r_job = copy.copy(p_job)
                    r_job.task = rep
                    r_job.core = rep.core
                    r_job.min_start_time = (r_job.deadline - r_job.WCET)
                    r_job.is_primary = False
                    r_job.shared_tasks.slist.append(r_job)
                    jobs.append(r_job)
                
                task.arrival += task.period
                task.deadline += task.period
                task.job_id += 1
    arrivals = sorted(list(set(arrivals)))
    return jobs, arrivals
                
def bounded_normal(min_val, max_val, center, spread_factor=0.25):
    range_val = max_val - min_val
    std_dev = spread_factor * range_val
    a, b = (min_val - center) / std_dev, (max_val - center) / std_dev
    return truncnorm.rvs(a, b, loc=center, scale=std_dev)


In [ ]:
# Arrival and dispatch
def manage_arrival(arrival_queue, cores, job_queue, time):

    new_arrived = False
    arr = -1
    if arrival_queue: arr = arrival_queue.popleft()
    if time == arr:
        found = False
        while not found:
            if not job_queue:
                break
            job = job_queue.popleft()
            if job.arrival != arr:
                job_queue.appendleft(job)
                found = True
                break
            if not job.is_primary:
                job.core.replica_st.append(job)
                job.core.replica_dl.append(job)
                job.core.ready.append(job)
            else:
                job.core.primary.append(job)
                job.core.ready.append(job)
            new_arrived = True
    else: arrival_queue.appendleft(arr)
# is_replica
    return new_arrived

def logic_of_replicas(c, time):
    if c.replica_dl[0].deadline <= c.replica_st[0].deadline and c.replica_dl[0].min_start_time <= c.replica_st[0].min_start_time and c.replica_dl[0].min_start_time<=time:
        job = c.replica_dl[0]
        c.running = job
    elif c.replica_dl[0].deadline >= c.replica_st[0].deadline and c.replica_dl[0].min_start_time >= c.replica_st[0].min_start_time and c.replica_dl[0].min_start_time<=time:
        job = c.replica_st[0]
        c.running = job
    elif c.replica_dl[0].deadline < c.replica_st[0].deadline and c.replica_dl[0].min_start_time > c.replica_st[0].min_start_time and c.replica_st[0].min_start_time<=time:
        job = c.replica_st[0]
        c.running = job
    elif c.replica_dl[0].deadline > c.replica_st[0].deadline and c.replica_dl[0].min_start_time < c.replica_st[0].min_start_time and c.replica_dl[0].min_start_time<=time:
        job = c.replica_dl[0]
        c.running = job

def dispatch(cores, time):

    for c in cores:
        # both are empty
        if (not c.primary) and (not c.replica_dl): continue
        # already a job is running
        if c.running:
            # replicas is empty
            if c.primary and (not c.replica_dl):
                if c.primary[0].deadline < c.running.deadline:
                    c.running = c.primary[0]
            # primary is empty
            elif (not c.primary) and c.replica_dl:
                if c.replica_dl[0].deadline < c.running.deadline and c.replica_dl[0].min_start_time <= time:
                    logic_of_replicas(c, time)

            else:
                if c.primary[0].deadline < c.running.deadline:
                    c.running = c.primary[0]

                if c.replica_dl[0].deadline < c.running.deadline and c.replica_dl[0].min_start_time <= time:
                    logic_of_replicas(c, time)

        # nothin is running
        else:
            # replicas is empty
            if c.primary and (not c.replica_dl): 
                c.running = c.primary[0]
            # primary is empty
            elif (not c.primary) and c.replica_dl:
                logic_of_replicas(c, time)
            # both not empty
            elif (c.primary[0].deadline <= c.replica_dl[0].deadline) or (c.replica_dl[0].min_start_time > time and c.replica_st[0].min_start_time > time):
                job = c.primary[0]
                c.running = job
            else:
                logic_of_replicas(c, time)


In [ ]:
# Dynamic voltage and frequency scaling

def DVFS(cores, time, vf_levels):
    for c in cores:
        if (c.current_utilization< 0): print(time ,c.current_utilization)
    for c in cores:
        if (not c.running) or c.running.vf_is_set: continue
        worst_lvl = c.running.worst_lvl
        if not c.running.is_primary:
            c.running.apply_DVFS(c.running.worst_v, c.running.worst_f)
            u_new = c.running.get_current_util()
            c.current_utilization += u_new
            c.running.task.current_job_id = c.running.job_id
            c.running_replica.append(c.running.task)
    
        elif c.current_utilization > 1:
            
            new_c_util = []
            for i in range(worst_lvl+1):
                ne = c.current_utilization + (c.running.task.exec_set[i]/c.running.task.period - c.running.get_current_util())
                if ne <= 1 and ne >=0 :
                    new_c_util.append((ne, i))
            # core util <= 1
            if new_c_util:
                new_c_util.sort(key=lambda t:t[0], reverse=True)
                lvl = new_c_util[0][1]
                u = c.running.get_current_util()
                c.running.apply_DVFS(vf_levels[lvl]['voltage'], vf_levels[lvl]['frequency'])
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
            # core util > 1
            # Update core utilization
            else:
                u = c.running.get_current_util()
                c.running.apply_DVFS(c.running.task.worst_v, c.running.task.worst_f)
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
                   
        else:
            new_c_util = []
            for i in range(worst_lvl+1):
                ne = c.current_utilization + (c.running.task.exec_set[i]/c.running.task.period - c.running.get_current_util())
                if ne <= 1 and ne>=0:
                    new_c_util.append((ne, i))

            # core util <= 1
            
            for i in range(len(new_c_util)):
                new_c_util.sort(key=lambda t:t[0], reverse=True)
                lvl = new_c_util[i][1]
                slack_covered = (c.running.task.initial_exec_task_time / (vf_levels[lvl]['frequency']/c.running.task.init_f)) - c.running.task.current_exec_time
                if c.running.task.slack - slack_covered < 0:
                    continue
                c.running.task.slack -= slack_covered
                u = c.running.get_current_util()
                c.running.apply_DVFS(vf_levels[lvl]['voltage'], vf_levels[lvl]['frequency'])
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
                break
            # core util > 1
            # Update core utilization
            else:
                u = c.running.get_current_util()
                c.running.apply_DVFS(c.running.task.worst_v, c.running.task.worst_f)
                u_new = c.running.get_current_util()
                c.current_utilization += (u_new - u)
            
        c.running.vf_is_set = True
        c.running.real_exec = c.running.current_exec_time
        c.running.is_real_exec_set = True


In [ ]:
# Execution and completion handling
def execute(cores, time, power_trace, util_trace, total_jobs, total_finished_jobs, faulty_jobs_counter, hyper_period):
    if total_jobs + faulty_jobs_counter == total_finished_jobs and time>=hyper_period: return
    u_tup = [0 for c in cores]
    for c in cores:
        if not c.running: continue # core is idle
        job = c.running
        if job.start_time == -1: job.start_time = time - 1
        job.real_exec -= 1

        if random.random() < 0.1:
            power_trace[c.id].append(job.get_peak_p())
        #     power_trace[c.id].append(p)
        else:    
            power_trace[c.id].append(bounded_normal(job.get_min_p(), job.get_peak_p(), job.get_avg_p()))
        u_tup[c.id] = (c.current_utilization)

    # power_trace.append(tuple(pt_tup))
    util_trace.append(tuple(u_tup))
    
def handle_finished_jobs(cores, time, TSP):
    TSP_violate_counter = 0
    total_finished_jobs = 0
    deadline_missed_counter = 0
    faulty_jobs_counter = 0

    utils = []
    for c in cores:
        utils.append(c.current_utilization)
        if (not c.running) or (c.running.real_exec>0): continue # core is idle or job is not done yet
        job = c.running
        job.finish_time = time

        if job.get_peak_p() > TSP:
            TSP_violate_counter += 1
        total_finished_jobs += 1

        if job.deadline < time:
            deadline_missed_counter += 1
        if job.fault:
            faulty_jobs_counter +=1
            if not job.is_primary:
                job.core.remove_replica(job)
                job.shared_tasks.slist.remove(job)
            else:
                job.core.remove_primary(job)
                job.shared_tasks.slist.remove(job)
            continue

        if job.is_primary:
            
            for rep in job.shared_tasks.slist:
                if rep.is_primary: continue
                exs = rep.core.remove_replica(rep)
                if exs:
                    pass
                else:
                    if (rep.task in rep.core.running_replica) and (rep.job_id > rep.task.current_job_id):
                        rep.core.running_replica.remove(rep.task)
                        rep.core.current_utilization -= rep.get_current_util()
            c.remove_primary(job)
            c.running = None
        else:
            for rep in job.shared_tasks.slist:
                if rep.is_primary: 
                    rep.core.remove_primary(rep)
                    continue
                exs = rep.core.remove_replica(rep)
                if exs:
                    pass
                else:
                    if (rep.task in rep.core.running_replica) and (rep.job_id > rep.task.current_job_id):
                        rep.core.running_replica.remove(rep.task)
                        rep.core.current_utilization -= rep.get_current_util()

    return TSP_violate_counter, total_finished_jobs, deadline_missed_counter, faulty_jobs_counter

def inject_fault(cores):
    available_core = []
    for c in cores:
        if c.ready:
            available_core.append(c)
    if not available_core: return
    c = random.choice(available_core)

    job = random.choice(c.ready)
    job.fault = True


In [ ]:
def simulate(env, data, TSP, hyper_period, output_path, log_path, total_jobs):
    cores = [Core(c) for c in range(env.cores)]
    data = sorted(data, key=lambda d:d.get_util(), reverse=True)
    ustate = {i: 0 for i in range(env.cores)}
    real_ustate = {i: 0 for i in range(env.cores)}
    map(data, ustate, env, cores)
    print(f"{cores[0].current_utilization}, {cores[1].current_utilization}, {cores[1].current_utilization}, {cores[3].current_utilization}")

    system = []
    #     system.append(total_task_reliability(tup))
    
    jobs, arrivals = create_jobs(data, hyper_period, cores, env)

# Calculate slack time
    calculate_slack(cores, hyper_period)
    define_slack(cores, data, hyper_period)

    #     # print(j.name, j.job_id, j.task)
    #         slacks.append(j.slack)
    
    arrival_queue = deque(arrivals)

    job_queue = deque(sorted(jobs, key=lambda j:(j.arrival, j.deadline)))

    new_arrived = False
    power_trace = [[] for c in cores]
    util_trace = []
    slack_trace = [[] for c in cores]
    TSP_violate_counter = 0
    total_finished_jobs = 0
    deadline_missed_counter = 0
    faulty_jobs_counter = 0
    do_fault = -1

# Runtime simulation
    for time in range(int(hyper_period*2 + 1)):

# Record per-core activity through the hyperperiod
        if not (total_jobs + faulty_jobs_counter == total_finished_jobs and time>=hyper_period):
            for c in cores: 
                if c.running:
                    slack_trace[c.id].append(1)
                else:
                    slack_trace[c.id].append(0)
        # # print(slack_trace)
        #         s_tr[c.id] = 1
        # slack_trace.append(s_tr)
        # # print(slack_trace)
        execute(cores, time, power_trace, util_trace, total_jobs, total_finished_jobs, faulty_jobs_counter, hyper_period)
        
        rv1, rv2, rv3, rv4 = handle_finished_jobs(cores, time, TSP)
        TSP_violate_counter += rv1
        total_finished_jobs += rv2
        deadline_missed_counter += rv3
        faulty_jobs_counter += rv4
        new_arrived = manage_arrival(arrival_queue, cores, job_queue, time)
        if new_arrived:
            for core in cores:
                core.EDF()

        dispatch(cores, time)
        DVFS(cores, time, env.vf_levels)
        utils = []

    print('\n=============================\n')
    print(TSP_violate_counter,
        total_finished_jobs,
        deadline_missed_counter,
        faulty_jobs_counter)
    print('\n=============================\n')
#     # for core in cores:
#     #     print(time, core.current_utilization, (core.running.name, core.running.job_id, 
#     #     core.running.rep_id, core.running.f, core.running.f_policy, core.running.worst_f,
#     #      core.running.init_f), core.running.real_exec, core.running.exec_set[0], core.running.current_exec_time,core.running.exec_set[4])
#     # print('\n----------------------------\n')

# # """
#     # print(arrivals)
    D_main =[]
    f_main =[]
    for d in jobs:
            if d.finish_time !=-1:
                D_main.append(d.deadline)
                f_main.append(d.finish_time)
    P = [1 for _ in range(len(f_main))]
    qos, UF = qos_system_no_backup(P, f_main, D_main)
                
    file_name = output_path + 'task_detail.csv'    
    with open(file_name, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Name", "Job_id","core_id","Arrival", "StartTime", "FinishTime", "Deadline", "Period", "v", "f", "reliability", "peak_p", "avg_p", "min_p"])
        for j in jobs:
            writer.writerow([j.name, j.job_id, j.core.id+1, j.arrival, j.start_time, j.finish_time, j.deadline, j.period, j.v, j.f, j.reliability, j.get_peak_p(), j.get_avg_p(), j.get_min_p()])

    energy = 0
    file_name = output_path + 'power_energy.csv'
    with open(file_name, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["Core1 load(%)","Core1 peak","Core1 P", "Core1 E","Core2 load(%)","Core2 peak", "Core2 P", "Core2 E","Core3 load(%)","Core3 peak","Core3 P", "Core3 E","Core4 load(%)","Core4 peak","Core4 P", "Core4 E"])
        tr = []
        for c in cores:
            tr.append(round((sum(slack_trace[c.id])/len(slack_trace[c.id]))*100, 2))
            tr.append(max(power_trace[c.id]))
            avg = sum(power_trace[c.id])/len(power_trace[c.id])
            tr.append(avg)
            tr.append(avg*hyper_period*1e-3)
            energy += avg*hyper_period*1e-3
        writer.writerow(tr)
        print(tr)
    ENERGY = 0
    for j in jobs:
        if j.is_real_exec_set: ENERGY += (j.current_exec_time-j.real_exec) * (1e-3)* j.get_avg_p()

    file_name = output_path + 'information.csv'
    with open(file_name, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(["ENERGY", "Energy" ,"QoS(%)"])
        writer.writerow([round(ENERGY, 2), round(energy, 2), round(qos*100, 3)])
        print([round(ENERGY, 2), round(energy, 2), round(qos*100, 3)])
        # writer.writerow([j.real_exec, j.current_exec_time, j.is_real_exec_set])
    # with open(file_name, mode='w', newline='') as file:
    #     writer.writerow(["Core 1", "Core 2", "Core 3", "Core 4"])
    #             tr.append(round(current_util[i], 3))
    #         writer.writerow(tr)

    # with open(file_name, mode='w', newline='') as file:
    #     writer.writerow(["ENERGY,Energy ,QoS(%)", "TSP Met Ratio(%)", "System reliability"])
    #     writer.writerow([round(qos*100, 3), round(((1-TSP_violate_counter/total_finished_jobs)*100), 3), system_r])
    # # return qos, TSP_violate_counter, total_finished_jobs, deadline_missed_counter, faulty_jobs_counter


In [ ]:
# Full baseline sweeps are opt-in because they can run for several hours.
RUN_BASELINE = False

if RUN_BASELINE:
    raise NotImplementedError(
        "Select load points from data/generated and call simulate(...) with an artifacts/ output path."
    )
